# Lab 4A: An agent with three tools (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 4 | about 10 minutes

- Extend the Day 3 agent with a third tool: read-only SQL over a bookings table
- Let a planner pick the next tool, one step at a time, with guardrails
- Solve a question that needs documents, then SQL, then arithmetic

## 1. Tools

In [ ]:
import os, re, glob, json, ast, operator, sqlite3, warnings
import numpy as np
from fastembed import TextEmbedding
from llm_client import call_llm, llm_available, llm_mode
warnings.filterwarnings("ignore")

DOCS = []
for f in sorted(glob.glob("data/corpus/*.txt")):
    head, body = open(f, encoding="utf-8").read().split("\n\n", 1)
    meta = dict(l.split(": ", 1) for l in head.splitlines() if ": " in l)
    DOCS.append(dict(title=meta["title"], text=body.strip()))
CHUNKS = []
for d in DOCS:
    w = re.findall(r"\S+\s*", d["text"])
    CHUNKS += [dict(title=d["title"], text="".join(w[i:i + 150]).strip()) for i in range(0, len(w), 150)]
_model = TextEmbedding("BAAI/bge-small-en-v1.5")
def embed(texts):
    v = np.array(list(_model.embed(list(texts))), dtype=float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)
CVEC = embed([c["text"] for c in CHUNKS])

def search_docs(query, k=3):
    """Top-k 150-word chunks by cosine similarity, each with its source title and score."""
    sims = CVEC @ embed([query])[0]
    top = np.argsort(-sims)[:k]
    return "\n".join(f"[{CHUNKS[i]['title']}] (sim {sims[i]:.2f}) {CHUNKS[i]['text']}" for i in top)

# In-memory bookings table (the London hotel cap in the policy corpus is GBP 210 per night)
DB = sqlite3.connect("file:bookings?mode=memory&cache=shared", uri=True)
DB.execute("CREATE TABLE bookings(city TEXT, traveller TEXT, nights INTEGER, price_per_night REAL, currency TEXT)")
DB.executemany("INSERT INTO bookings VALUES (?,?,?,?,?)", [
    ("London", "A. Shah", 3, 195, "GBP"), ("London", "B. Okafor", 2, 240, "GBP"),
    ("London", "C. Meyer", 4, 210, "GBP"), ("London", "D. Rossi", 3, 265, "GBP"),
    ("London", "E. Tan", 1, 180, "GBP"), ("London", "F. Novak", 2, 225, "GBP"),
    ("Paris", "G. Lopez", 3, 250, "EUR"), ("Paris", "H. Kim", 2, 230, "EUR"),
    ("New York", "I. Brown", 4, 360, "USD"), ("New York", "J. Singh", 2, 320, "USD"),
    ("Manchester", "K. Evans", 2, 140, "GBP"), ("Dublin", "L. Byrne", 3, 215, "EUR")])

def query_sql(sql, limit=20):
    """Read-only: one SELECT statement, at most `limit` rows. Returns JSON rows."""
    s = sql.strip().rstrip(";")
    if ";" in s or not re.match(r"(?i)^select\b", s): raise ValueError("only a single SELECT statement is allowed")
    if re.search(r"(?i)\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b", s): raise ValueError("write keywords are blocked")
    cur = DB.execute(s)
    cols = [c[0] for c in cur.description]
    return json.dumps([dict(zip(cols, r)) for r in cur.fetchmany(limit)])

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}
def calculator(expression):
    """Safe arithmetic only: numbers and + - * / ** %."""
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only plain arithmetic is allowed")
    return str(round(ev(ast.parse(expression, mode="eval").body), 4))

TOOLS = {"search_docs": (search_docs, "search the company documents; input = a search query"),
         "query_sql": (query_sql, "read-only SQL (one SELECT) on bookings; input = a SELECT statement"),
         "calculator": (calculator, "evaluate arithmetic such as 210 * 3; input = an expression")}
SCHEMA = "bookings(city TEXT, traveller TEXT, nights INTEGER, price_per_night REAL, currency TEXT)"
print(calculator("12 * 3"), "|", query_sql("SELECT COUNT(*) AS n FROM bookings"), "|", list(TOOLS))

## 2. Planner
Free LLM? Set `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL` (same as Day 2) in the commented cell below, run it, then run the planner cell.
No key: an OFFLINE rule-based stand-in plans instead (it is not a language model).

In [ ]:
# os.environ["LLM_API_KEY"]  = "PASTE-YOUR-KEY"
# os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # Google AI Studio: https://generativelanguage.googleapis.com/v1beta/openai/   Ollama: http://localhost:11434/v1
# os.environ["LLM_MODEL"]    = "PASTE-A-MODEL-NAME"

In [ ]:
NUM = r"\d[\d,]*\.?\d*"
def _num(s): return float(s.replace(",", ""))

def plan_offline(question, scratch):
    """OFFLINE rule-based stand-in for the LLM planner (demo questions only)."""
    q, done = question.lower(), [s["action"] for s in scratch]
    m = re.search(rf"({NUM})\s*(?:%|percent)\s*of\s*({NUM})", q)
    if m and not scratch:
        return dict(thought="Pure arithmetic, no documents needed.", action="calculator", input=f"{_num(m.group(1)):g} / 100 * {_num(m.group(2)):g}")
    if "booking" in q and "cap" in q:
        if "search_docs" not in done:
            return dict(thought="I need the policy hotel cap first.", action="search_docs", input="hotel cap per night London")
        cap = re.search(r"London\s+(?:GBP|EUR|USD)\s+(\d+)", scratch[0]["observation"])
        if not cap: return dict(thought="The cap was not in the search results.", action="finish", input="")
        if "query_sql" not in done:
            c = cap.group(1)
            return dict(thought=f"The cap is {c}. Now find London bookings above it.", action="query_sql",
                        input=f"SELECT traveller, nights, price_per_night FROM bookings WHERE city='London' AND currency='GBP' AND price_per_night > {c} ORDER BY traveller")
        if "calculator" not in done:
            rows, c = json.loads(scratch[1]["observation"]), cap.group(1)
            expr = " + ".join(f"{r['nights']} * ({r['price_per_night']:g} - {c})" for r in rows) or "0"
            return dict(thought="Total overage = sum of nights * (price - cap).", action="calculator", input=expr)
        return dict(thought="I have the total overage.", action="finish", input="")
    if not scratch:
        return dict(thought="Try the documents first.", action="search_docs", input=question)
    return dict(thought="I have the calculation result." if scratch[-1]["action"] == "calculator" else "Nothing useful found.", action="finish", input="")

def plan_llm(question, scratch):
    tools = "\n".join(f"- {n}: {d}" for n, (_, d) in TOOLS.items()) + "\n- finish: give the final answer"
    pad = "\n".join(f"Step {i}: {s['action']}({s['input']}) -> {s['observation'][:400]}" for i, s in enumerate(scratch, 1)) or "(empty)"
    prompt = (f"Tools:\n{tools}\n\nTable: {SCHEMA}\n\nScratchpad:\n{pad}\n\nQUESTION: {question}\n"
              'Reply with JSON only: {"thought": "...", "action": "<tool name or finish>", "input": "..."}')
    try:
        out = json.loads(call_llm(prompt, system="You are a careful planner. Pick ONE next action.", json_mode=True))
        if out.get("action") in list(TOOLS) + ["finish"]: return dict(thought=out.get("thought", ""), action=out["action"], input=str(out.get("input", "")))
        return dict(thought=f"Planner chose a disallowed action: {out.get('action')!r}", action="finish", input="")
    except Exception as e:
        return dict(thought=f"Planner output could not be parsed ({type(e).__name__})", action="finish", input="")

def plan(question, scratchpad):
    return plan_llm(question, scratchpad) if llm_available() else plan_offline(question, scratchpad)

print("LLM:", llm_mode())
print("=== PLANNER IN USE:", "LLM planner (JSON mode)" if llm_available() else "OFFLINE rule-based stand-in (not a language model)", "===")

## 3. Agent loop with guardrails
Guardrails: at most 5 steps, allow-listed tools only, SELECT-only SQL, every step printed.

In [ ]:
MAX_STEPS = 5
def run_agent(question):
    scratch = []
    print(f"\nTASK: {question}")
    for step in range(1, MAX_STEPS + 1):
        p = plan(question, scratch)
        print(f"\n[{step}] Thought: {p['thought']}\n    Action: {p['action']}({p['input']})")
        if p["action"] == "finish": break
        if p["action"] not in TOOLS: print("    blocked: tool not on allow-list"); break
        try: obs = TOOLS[p["action"]][0](p["input"])
        except Exception as e: obs = f"ERROR: {e}"
        print("    Observation:", obs[:160].replace("\n", " ") + ("..." if len(obs) > 160 else ""))
        scratch.append(dict(p, observation=obs))
    return final_answer(question, scratch)

def final_answer(question, scratch):
    by = {s["action"]: s for s in scratch if not s["observation"].startswith("ERROR")}
    if llm_available():
        pad = "\n".join(f"{s['action']}({s['input']}) -> {s['observation'][:600]}" for s in scratch)
        return call_llm(f"{pad}\n\nQUESTION: {question}", system="Answer only from the scratchpad; if it lacks the answer say 'I could not find this'.")
    if "query_sql" in by and "calculator" in by:
        cap = re.search(r"London\s+GBP\s+(\d+)", by["search_docs"]["observation"]).group(1)
        rows = json.loads(by["query_sql"]["observation"])
        lines = "\n".join(f"  - {r['traveller']}: {r['nights']} nights at GBP {r['price_per_night']:g}" for r in rows)
        return f"{len(rows)} London bookings exceed the GBP {cap} cap:\n{lines}\nTotal over the cap: GBP {float(by['calculator']['observation']):g}."
    if "calculator" in by: return f"The result is {float(by['calculator']['observation']):,g}."
    return "I could not find this in the provided documents or the bookings table."

## 4. Three tasks

In [ ]:
for task in ["Which London bookings exceed the hotel cap, and how much over in total?",
             "What is 12.5% of 8,400?",
             "Who won the 1998 world cup?"]:
    print("FINAL ANSWER:", run_agent(task))

**Recap**
- Tool choice is agentic: the planner decides each next step from the scratchpad, not a fixed pipeline.
- Guardrails: 5-step cap, allow-listed tools, SELECT-only single-statement SQL with a row limit, safe calculator.
- Offline, the planner is a rule-based stand-in; with a key, the LLM chooses each step.
- Day 4 next: MCP, a standard way to plug tools like these into any agent.